### Courtesy of unsloth, transformer, and llama

<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://github.com/huggingface/transformers"><img src="https://i-blog.csdnimg.cn/blog_migrate/7d518bf681de868a985cf0137266dcdf.png" width="145"></a>
<a href="https://www.llama.com/"><img src="https://user-images.githubusercontent.com/1991296/230134379-7181e485-c521-4d23-a0d6-f7b3b61ba524.png" width="125"></a></a>

### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl==0.15.2 triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 190711,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2", # 3.1
)

# --- ✨ New: Prepare Conversations BEFORE anything else ---
def prepare_conversations(examples):
    prompts = examples["Prompt"]
    responses = examples["Response"]

    conversations = []
    for prompt, response in zip(prompts, responses):
        convo = [
            { "from": "user", "value": prompt },
            { "from": "assistant", "value": response }
        ]
        conversations.append(convo)

    return { "conversations": conversations }
# --- End new block ---

#from datasets import load_dataset
dataset = load_dataset("csv", data_files="/content/drive/MyDrive/Liahona-GPT/Dataset/Liahona_QA_guardrails_0627_enriched.csv", split="train")


### Data formating

In [ ]:
#from datasets import load_dataset
# Step 1: Load CSV
#dataset = load_dataset("csv", data_files="/content/drive/MyDrive/Liahona-GPT/Dataset/Liahona_QA_guardrails_0627_enriched.csv", split="train")

# Step 2: Prepare Conversations



from datasets import load_dataset, Dataset
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Liahona-GPT/Dataset/Liahona_QA_guardrails_0806_Jusus_is_Christ.csv")
# Convert to HF dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(prepare_conversations, batched=True)


In [ ]:
dataset['conversations'][0]

In [ ]:
dataset.column_names

In [ ]:
# Step 3: Standardize ShareGPT (make sure roles are correct etc.)
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)

# Step 4: Apply chat template formatting
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=["Prompt", "Response", "Label", "conversations"])


We look at how the conversations are structured for item 5:

In [ ]:
dataset.column_names

And we see how the chat template transformed these conversations.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

<a name="Train"></a>
### Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),

    dataset_num_proc = 2,
    packing = False, # No packing
    args = TrainingArguments(
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        num_train_epochs = 4, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 5e-6,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
trainer_stats = trainer.train()

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "LDS missionary", "content": "Tell me about the Book of Mormon."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.1, min_p = 0.8)

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
from google.colab import userdata
# Load your Hugging Face token from Colab secrets
HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
userdata.get('HUGGINGFACE_TOKEN')
model_path = "bigrainlin/liahona-GPT-CoLAB_0806_conversation"

In [ ]:
# Merge to 16bit
model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged(model_path, tokenizer, save_method = "merged_16bit", token = HF_TOKEN)

# Merge to 4bit
#model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
#model.push_to_hub_merged(model_path, tokenizer, save_method = "merged_4bit", token = HF_TOKEN)

# Just LoRA adapters
#model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
#model.push_to_hub_merged(model_path, tokenizer, save_method = "lora", token = HF_TOKEN)

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:

# # Save to 8bit Q8_0
# if False: model.save_pretrained_gguf("model", tokenizer,)
# # Remember to go to https://huggingface.co/settings/tokens for a token!
# # And change hf to your username!
# if False: model.push_to_hub_gguf(model_path, tokenizer, token = HF_TOKEN)
#
# # Save to 16bit GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
# if False: model.push_to_hub_gguf(model_path, tokenizer, quantization_method = "f16", token = HF_TOKEN)
#
# # Save to q4_k_m GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
# if False: model.push_to_hub_gguf(model_path, tokenizer, quantization_method = "q4_k_m", token = HF_TOKEN)

# Save to multiple GGUF options - much faster if you want multiple!
if True:
    model.push_to_hub_gguf(
        model_path, # Change hf to your username!
        tokenizer,
        quantization_method = ["q8_0",],
        token = HF_TOKEN, # Get a token at https://huggingface.co/settings/tokens
    )

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install nbformat nbconvert
import nbformat as nbf, pathlib
nb_path = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/Llama3.2_(3B)-Liahona_Training_Conversational.ipynb")  # ← update
with nb_path.open("r", encoding="utf-8") as f: nb = nbf.read(f, as_version=nbf.NO_CONVERT)
nb.metadata.pop("widgets", None)
with nb_path.open("w", encoding="utf-8") as f: nbf.write(nb, f)
!jupyter nbconvert --ClearOutputPreprocessor.enabled=True --to notebook --inplace "{nb_path}"
print("Cleaned:", nb_path)
